In [ ]:
#| default_exp machine_learning.context_extraction

In [ ]:
#| export
from typing import Optional

import lmstudio as lms
from lmstudio import LLM

from trouver.notation.glossary import _resolve_info_notes_for_index
from trouver.obsidian.vault import VaultNote
from trouver.obsidian.file import MarkdownFile
from trouver.personal_vault.note_processing import process_standard_information_note

from trouver.llm_core.call_llm import call_llm, process_llm_response, smart_truncate, SupportedLLM

import re
from typing import List, Optional, Dict, Callable, Tuple, Any
from pathlib import Path

In [ ]:
from unittest.mock import patch

import time
from unittest.mock import MagicMock, patch
from fastcore.test import *

In [ ]:
#| export

# Assuming the client is imported from your established lmstudio module
# from lmstudio.client import LMStudio 

# --- Constants for the LLM ---
CONTEXT_EXTRACTION_SYSTEM_PROMPT = r"""
### Role
You are a Mathematical Secretary and Pre-Processor. Your goal is to read mathematical prose and extract the "Ambient Environment" before any formal statements (Propositions/Theorems) are encountered. You are preparing a "Silver Platter" of definitions so that a reader can understand future formal statements without looking back. You will be provided with a 'Previous Context' (existing definitions) and a 'New Passage'. Your task is to update the 'Silver Platter' by identifying only what has changed or been added.

### Extraction Focus
1. **Persistent Notational Conventions**: Identify ONLY symbols introduced for the first time. If a symbol exists in 'Previous Context', it is forbidden to list it under 'Primary Objects'.
2. **Ambient Assumptions**: Extract the new properties assigned to the objects currently under discussion (e.g., "assume all manifolds are smooth," "let $X$ be a compact space").
3. **Strict Non-Redundancy**: If a passage provides new information about an object already in 'Previous Context' (e.g., "K is now assumed to be finite"), move this information to Global Constraints or Defined Relations. Do not re-instantiate the object.
4. **Implicit Structural Relations**: Identify how objects relate to each other as established in the prose (e.g., $A \subseteq B$ is a ring extension, $G$ acts on $X$).
5. **Scope of Validity**: Note if the author explicitly limits the discussion (e.g., "Throughout this section, we assume $n > 2$").
6. The Null Case: If the passage contains no new notations or assumptions beyond what is in the 'Previous Context', output the 'Status' message below. If the passage is purely expository, motivational, or repeats 'Previous Context' without adding new constraints/notations, you must only output the Status message. Do not include empty headers or "None new" lists.

### Input Structure
You will receive input in the following format:
* **PREVIOUS CONTEXT**: A summary of mathematical objects and notations already established.
* **NEW PASSAGE**: The raw text to be analyzed for new additions.

### Output Format (The Silver Platter)
If the text contains relevant information, use the following structure:

**New Contextual Updates:**
* **Primary Objects**: [e.g., $K$ (Algebraic number field), $V$ (Vector space)]
* **Defined Relations**: [e.g., $A$ is a subring of $B$]
* **Active Notations**: [e.g., $S$ = set of prime ideals, $Cl^S_K$ = S-class group]
* **Global Constraints**: [e.g., "All rings are assumed to be Noetherian," "Characteristic of $K$ is 0"]

**If no relevant information is found:**
* **Status**: No new contextual or notational instantiations identified in this passage.

Only include a bullet point if there is information to populate it. If a specific section (e.g., Active Notations) has no updates but others do, omit that bullet point entirely.

---
### Technical Directives
* Prioritize the accuracy of symbol extraction over lengthy prose explanations.
* Use LaTeX for all mathematical symbols. Generally maintain the LaTeX convention of the original text except to fix typos in the original text.
* Prohibit Meta-talk: Do not include "Notes," "Explanations," or parenthetical justifications (e.g., "(already defined)"). The Silver Platter must contain only the mathematical data.
* Property Mapping: Treat adjectives (e.g., "compact," "Noetherian," "flat") as Global Constraints or Defined Relations, never as Primary Objects.
"""


In [ ]:
#| export
# In context_extraction module
def extract_context_with_lm(
    model: SupportedLLM,
    excerpt_text: str,
    max_context: int = 4096,
    system_prompt: str = CONTEXT_EXTRACTION_SYSTEM_PROMPT,
    config: Optional[dict] = None,
    verbose: bool = True,
    return_usage: bool = False  # <-- Add this parameter
) -> Optional[str] | Tuple[Optional[str], Any]: # <-- Update return hint
    
    # Context extraction usually needs more 'room' for the input text
    reserved = 1200 
    truncated_text = smart_truncate(excerpt_text, model, max_context, reserved)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": truncated_text}
    ]

    raw_res = call_llm(
        model, messages, config, verbose,
        return_usage=return_usage)

    # if return_usage:
    #     raw_output, usage = raw_res
    #     clean_content = process_llm_response(raw_output, return_thoughts=False)
    #     return clean_content, usage

    if return_usage:
        raw_output, usage = raw_res
        clean_content = process_llm_response(raw_output, return_thoughts=False)
        return clean_content, usage # This is 2 values.
    # We use the 'clean' version in case the model starts thinking
    return process_llm_response(raw_res, return_thoughts=False)

In [ ]:
#| hide
@patch('__main__.call_llm')
@patch('__main__.process_llm_response')
def test_usage_passthrough(mock_process, mock_call):
    mock_model = MagicMock()
    mock_usage = {"total_tokens": 100}
    
    # Mock call_llm returning the tuple (content, usage)
    mock_call.return_value = ("Raw Output", mock_usage)
    mock_process.return_value = "Clean Content"
    
    # Execute with return_usage=True
    content, usage = extract_context_with_lm(
        mock_model, "Some text", return_usage=True, verbose=False
    )
    
    test_eq(content, "Clean Content")
    test_eq(usage["total_tokens"], 100)
    # Verify call_llm received the flag
    test_eq(mock_call.call_args[1]['return_usage'], True)

test_usage_passthrough()

In [ ]:
#| export
from typing import List, Optional
from pathlib import Path


# --- Assumed Imports from Your Library ---
#
#
#   from .glossary import _resolve_info_notes_for_index

def _generate_context_extraction_markdown(
        info_notes: List[VaultNote],
        model: SupportedLLM,
        system_prompt: str,
        ) -> str:
    """
    Generates Markdown by extracting context from a list of info notes.
    """
    markdown_lines = []
    for note in info_notes:
        print(f"  - Processing '{note.name}' for context...")
        mf = MarkdownFile.from_vault_note(note)
        note_text = str(process_standard_information_note(mf, note.vault))
        
        # This call now correctly uses your internal lmstudio library via the updated function
        context_info = extract_context_with_lm(
            model,
            note_text, 
        )
        
        if context_info:
            markdown_lines.append(f"- [[{note.name}]]")
            context_lines = context_info.split('\n')
            markdown_lines.append(f"    - {context_lines[0]}")
            for line in context_lines[1:]:
                markdown_lines.append(f"      {line}")
                
    return "\n".join(markdown_lines)

In [ ]:
#| export
import re
from typing import List, Optional, Dict
from pathlib import Path

# --- Assumed Imports from Your Library ---
# from trouver.obsidian.vault import VaultNote
# from trouver.obsidian.markdown.file import MarkdownFile
# from trouver.obsidian.markdown.processing import process_standard_information_note
# from .glossary import _resolve_info_notes_for_index
# from .llm import extract_context_with_lm, LLM, CONTEXT_EXTRACTION_SYSTEM_PROMPT

def _parse_existing_context_note(note_text: str) -> Dict[str, str]:
    """
    Parses an existing context extraction note into a dictionary.
    
    Args:
        note_text: The content of the markdown note.
        
    Returns:
        A dictionary where keys are note names and values are the context strings.
    """
    data = {}
    current_note_name = None
    current_lines = []
    
    # Regex to match "- [[Note Name]]"
    note_header_pattern = re.compile(r'^\s*-\s*\[\[(.*?)\]\]')
    
    for line in note_text.split('\n'):
        match = note_header_pattern.match(line)
        if match:
            # If we were processing a note, save it before starting the new one
            if current_note_name:
                data[current_note_name] = "\n".join(current_lines).strip()
            
            current_note_name = match.group(1)
            current_lines = []
        elif current_note_name:
            # This line belongs to the current note.
            # We strip the list formatting (indentation + dash) to get the raw text.
            # Assuming format: "    - Text" or "      Text"
            stripped = line.strip()
            if stripped.startswith('- '):
                stripped = stripped[2:]
            current_lines.append(stripped)
            
    # Save the last note
    if current_note_name:
        data[current_note_name] = "\n".join(current_lines).strip()
        
    return data


In [ ]:
#| export
def _format_context_extraction(
    info_notes: List[VaultNote], 
    context_data: Dict[str, str]
) -> str:
    """
    Formats the context dictionary back into the Markdown list structure,
    preserving the order of the provided info_notes list.
    """
    markdown_lines = []
    for note in info_notes:
        if note.name in context_data:
            context_text = context_data[note.name]
            if context_text:
                markdown_lines.append(f"- [[{note.name}]]")
                # Split multi-line context to format correctly
                lines = context_text.split('\n')
                # First line gets the bullet
                markdown_lines.append(f"    - {lines[0]}")
                # Subsequent lines get indentation
                for line in lines[1:]:
                    markdown_lines.append(f"      {line}")
    return "\n".join(markdown_lines)

In [ ]:
def test_format_preserves_order_and_content(self):
        """
        Verifies that formatting follows the info_notes order and handles multiline context.
        """
        info_notes = [self.note_b, self.note_a] # Note B then A
        context_data = {
            "Definition A": "Line 1\nLine 2",
            "Theorem B": "Single Line"
        }
        
        result = _format_context_extraction(info_notes, context_data)
        
        # Check B is first (per info_notes list)
        lines = result.split('\n')
        self.assertEqual(lines[0], "- [[Theorem B]]")
        self.assertEqual(lines[1].strip(), "- Single Line")
        
        # Check A follows
        self.assertIn("- [[Definition A]]", result)
        self.assertIn("      Line 2", result) # Check indentation of second line

In [ ]:
#| export

def _format_llm_input(previous_context: str, new_passage: str) -> str:
    """Formats the input for the LLM with previous context and new text."""
    # If there is no previous context, we can indicate that or leave it empty.
    context_str = previous_context if previous_context.strip() else "None."
    return f"PREVIOUS CONTEXT:\n{context_str}\n\nNEW PASSAGE:\n{new_passage}"

In [ ]:
#| export

# --- Assumed Imports ---
# from trouver.obsidian.vault import VaultNote
# from trouver.obsidian.markdown.file import MarkdownFile
# from trouver.obsidian.markdown.processing import process_standard_information_note
# from .glossary import _resolve_info_notes_for_index
# from .llm import extract_context_with_lm, LLM

# Updated Type Alias
# Inputs: 
#   1. The current VaultNote
#   2. The dictionary of all existing context data
#   3. The ordered list of all note names in the sequence
# Output: A list of note names (keys in the dictionary) to include
ContextSelector = Callable[[VaultNote, Dict[str, str], List[str]], List[str]]

def _default_context_selector(
    note: VaultNote, 
    current_data: Dict[str, str],
    ordered_note_names: List[str]
) -> List[str]:
    """
    Default behavior: Returns the context of the 5 most recent notes 
    that appear before the current note in the ordered list.
    """
    try:
        current_index = ordered_note_names.index(note.name)
    except ValueError:
        # Fallback: if note not found in list, return everything (unlikely)
        return [n for n in current_data.keys() if n != note.name]

    # Get all potential predecessors
    predecessors = ordered_note_names[:current_index]
    
    # Filter to keep only those that actually have data in current_data
    available_predecessors = [name for name in predecessors if name in current_data]
    
    # Return the last 5 (the most recent ones)
    return available_predecessors[-5:]


In [ ]:
#| hide
from fastcore.test import *

def test_sliding_window_selector():
    # Setup a dummy sequence of 10 notes
    all_names = [f"Note {i}" for i in range(10)]
    # Simulate that we have data for all of them
    current_data = {name: f"Context {name}" for name in all_names}
    
    mock_note = MagicMock()
    
    # Test Case 1: Early in the list (Note 2)
    mock_note.name = "Note 2"
    res = _default_context_selector(mock_note, current_data, all_names)
    test_eq(res, ["Note 0", "Note 1"]) # Only 2 predecessors available
    
    # Test Case 2: Middle of the list (Note 8)
    mock_note.name = "Note 8"
    res = _default_context_selector(mock_note, current_data, all_names)
    # Should get the 5 immediately preceding: 3, 4, 5, 6, 7
    test_eq(res, ["Note 3", "Note 4", "Note 5", "Note 6", "Note 7"])
    
    # Test Case 3: Missing data in middle
    del current_data["Note 5"]
    res = _default_context_selector(mock_note, current_data, all_names)
    # Should skip Note 5 because it has no context data
    test_eq(res, ["Note 2", "Note 3", "Note 4", "Note 6", "Note 7"])

test_sliding_window_selector()

In [ ]:

# #| export
# def create_context_extraction_for_index_note(
#     index_note: VaultNote,
#     model: 'SupportedLLM',
#     info_notes: Optional[List[VaultNote]] = None,
#     max_context: int = 4096,
#     system_prompt: Optional[str] = CONTEXT_EXTRACTION_SYSTEM_PROMPT,
#     config: Optional[dict] = None,
#     verbose: bool = False,
#     context_selector: Optional[ContextSelector] = None,
# ) -> None:
#     """
#     Generates and saves a context extraction file for a given index note.
    
#     Args:
#         index_note: The index note to process.
#         model: The LLM model object.
#         info_notes: Optional list of specific notes to process.
#         system_prompt: Optional system prompt override.
#         context_selector: A function that determines which previous contexts 
#                           should be included. Defaults to a sliding window of 5.
#     """
#     # 0. Set default selector if None
#     if context_selector is None:
#         context_selector = _default_context_selector

#     # 1. Resolve the Master List (File Structure)
#     all_associated_notes = _resolve_info_notes_for_index(index_note)
    
#     # Create the ordered list of names for the selector
#     all_note_names = [n.name for n in all_associated_notes]
    
#     # 2. Determine the Work Queue
#     target_notes_set = set(info_notes) if info_notes else set(all_associated_notes)

#     if not all_associated_notes:
#         print(f"Warning: No info notes found for '{index_note.name}'.")
#         return

#     # 3. Determine Context Note Path
#     prefix = "_index_"
#     interesting_name = index_note.name[len(prefix):] if index_note.name.startswith(prefix) else index_note.name
#     context_note_name = f"_context_extraction_{interesting_name}"
#     parent_dir = Path(index_note.rel_path).parent
#     context_rel_path = parent_dir / f"{context_note_name}.md"
    
#     context_note = VaultNote(index_note.vault, rel_path=str(context_rel_path))

#     # 4. Load Existing Data
#     existing_data = {}
#     if context_note.exists():
#         print(f"Found existing context note: {context_note.name}. Parsing...")
#         existing_data = _parse_existing_context_note(context_note.text())
#     else:
#         context_note.create()

#     # 5. Process Notes Sequentially
#     print(f"Processing sequence of {len(all_associated_notes)} info note(s)...")
    
#     current_data = existing_data.copy()
    
#     for i, note in enumerate(all_associated_notes):
#         # Determine if we should run the LLM for this note
#         should_run_llm = (note in target_notes_set) and (note.name not in current_data)
        
#         if should_run_llm:
#             print(f"[{i+1}/{len(all_associated_notes)}] Extracting context for '{note.name}'...")
            
#             # Prepare Text
#             mf = MarkdownFile.from_vault_note(note)
#             note_text = str(process_standard_information_note(mf, note.vault))
            
#             # --- NEW LOGIC: Select Context ---
#             # Pass the ordered list of names to the selector
#             relevant_note_names = context_selector(note, current_data, all_note_names)
            
#             selected_context_strings = []
#             for name in relevant_note_names:
#                 # Double check existence, though selector should handle it
#                 if name in current_data:
#                     selected_context_strings.append(current_data[name])
            
#             prev_context_str = "\n".join(selected_context_strings)
#             # ---------------------------------
            
#             # Format Input
#             llm_input = _format_llm_input(prev_context_str, note_text)
            
#             # Run Prediction
#             context_info = extract_context_with_lm(
#                 model, llm_input, max_context=max_context, system_prompt=system_prompt, config=config,verbose=verbose)
            
#             if context_info:
#                 # Update Data
#                 current_data[note.name] = context_info
                
#                 # Write to file immediately
#                 full_content = _format_context_extraction(all_associated_notes, current_data)
#                 context_note.write(full_content)
#             else:
#                 print(f"  -> No context returned for '{note.name}'.")

#     print(f"Context extraction completed: {context_note.name}")

In [ ]:
#| export
def _get_context_note_for_index(index_note: VaultNote) -> VaultNote:
    """Determines path and initializes the context extraction note."""
    prefix = "_index_"
    name = index_note.name
    interesting_name = name[len(prefix):] if name.startswith(prefix) else name
    context_note_name = f"_context_extraction_{interesting_name}"
    
    parent_dir = Path(index_note.rel_path).parent
    context_rel_path = parent_dir / f"{context_note_name}.md"
    
    context_note = VaultNote(index_note.vault, rel_path=str(context_rel_path))
    if not context_note.exists():
        context_note.create()
    return context_note


In [ ]:
#| export
# def _process_single_note_context(
#     note: VaultNote,
#     model: 'SupportedLLM',
#     current_data: Dict[str, str],
#     all_note_names: List[str],
#     context_selector: ContextSelector,
#     **kwargs
# ) -> Optional[str]:
#     """Handles the LLM logic and context assembly for a single note."""
#     mf = MarkdownFile.from_vault_note(note)
#     note_text = str(process_standard_information_note(mf, note.vault))
    
#     relevant_names = context_selector(note, current_data, all_note_names)
#     selected_strings = [current_data[n] for n in relevant_names if n in current_data]
    
#     llm_input = _format_llm_input("\n".join(selected_strings), note_text)
#     return extract_context_with_lm(
#         model, 
#         llm_input, 
#         max_context=kwargs.get('max_context', 4096), 
#         system_prompt=kwargs.get('system_prompt'), 
#         config=kwargs.get('config'),
#         verbose=kwargs.get('verbose', False)
#     )

#| export
def _process_single_note_context(
    note: VaultNote,
    model: 'SupportedLLM',
    current_data: Dict[str, str],
    all_note_names: List[str],
    context_selector: ContextSelector,
    **kwargs
) -> Optional[str] | Tuple[Optional[str], Any]: # Updated hint
    """Handles the LLM logic and context assembly for a single note."""
    mf = MarkdownFile.from_vault_note(note)
    note_text = str(process_standard_information_note(mf, note.vault))
    
    relevant_names = context_selector(note, current_data, all_note_names)
    selected_strings = [current_data[n] for n in relevant_names if n in current_data]
    
    llm_input = _format_llm_input("\n".join(selected_strings), note_text)
    
    # We must explicitly pass return_usage into extract_context_with_lm
    # and return whatever it gives us (either a string or a tuple)
    return extract_context_with_lm(
        model, 
        llm_input, 
        max_context=kwargs.get('max_context', 4096), 
        system_prompt=kwargs.get('system_prompt'), 
        config=kwargs.get('config'),
        verbose=kwargs.get('verbose', False),
        return_usage=kwargs.get('return_usage', False) # Ensure this is passed!
    )

In [ ]:
#| export
import time
import re



def _parse_reset_time(val) -> float:
    """Parses various rate limit reset formats into seconds."""
    if val is None: return 0.0
    s_val = str(val).strip()
    
    # Case A: Unix Timestamp (common in Anthropic)
    if s_val.replace('.', '').isdigit() and float(s_val) > 1700000000:
        return max(0.0, float(s_val) - time.time())
    
    # Case B: Duration string like "6m0s" or "1.5s" (OpenAI/Groq)
    if any(u in s_val for u in ['h', 'm', 's']):
        times = re.findall(r'(\d+(?:\.\d+)?)([hms])', s_val)
        total = 0.0
        for amt, unit in times:
            mult = {'h': 3600, 'm': 60, 's': 1}[unit]
            total += float(amt) * mult
        return total
        
    # Case C: Simple integer/float seconds
    try: return float(s_val)
    except: return 0.0


In [ ]:
#| export
#| export
def _get_sleep_time_from_usage(usage: dict, threshold_pct: float = 0.1, default_sleep: float = 0.0) -> float:
    """Calculates sleep time, prioritizing default_sleep unless limits are low."""
    wait_time = default_sleep
    if not usage or not isinstance(usage, dict): return wait_time

    # Normalize keys to lowercase for robustness
    u = {str(k).lower(): v for k, v in usage.items()}

    # 1. Determine Remaining and Limit (TPM then RPM)
    rem = u.get('x-ratelimit-remaining-tokens') or u.get('anthropic-ratelimit-tokens-remaining') or u.get('x-ratelimit-remaining')
    lim = u.get('x-ratelimit-limit-tokens') or u.get('anthropic-ratelimit-tokens-limit') or u.get('x-ratelimit-limit')
    
    # 2. Determine Reset (Crucial: matching the specific anthropic key from test)
    res = (u.get('anthropic-ratelimit-tokens-reset') or 
           u.get('x-ratelimit-reset-tokens') or 
           u.get('anthropic-ratelimit-requests-reset') or
           u.get('x-ratelimit-reset'))

    try:
        if rem is not None and lim is not None:
            rem_val, lim_val = float(rem), float(lim)
            if (rem_val / lim_val) < threshold_pct:
                dynamic_wait = _parse_reset_time(res)
                # Ensure we add the buffer to the dynamic wait, then take max against default
                wait_time = max(wait_time, dynamic_wait + 0.5)
    except (ValueError, TypeError):
        pass

    return wait_time

In [ ]:
#| export
#| export
def create_context_extraction_for_index_note(
    index_note: VaultNote,
    model: 'SupportedLLM',
    info_notes: Optional[List[VaultNote]] = None,
    max_context: int = 4096,
    system_prompt: Optional[str] = CONTEXT_EXTRACTION_SYSTEM_PROMPT,
    config: Optional[dict] = None,
    verbose: bool = False,
    context_selector: Optional[ContextSelector] = None,
    sleep_threshold: float = 0.1,
    default_sleep: float = 1.0
) -> None:
    """
    Creates/updates a context-extraction note for a given index note.
    Includes rate-limit aware throttling and default sleep pacing.
    """
    context_selector = context_selector or _default_context_selector
    all_associated = _resolve_info_notes_for_index(index_note)
    
    if not all_associated:
        print(f"Warning: No info notes found for '{index_note.name}'.")
        return

    # Filter to specific notes if requested, otherwise process all associated
    target_set = set(info_notes) if info_notes else set(all_associated)
    
    context_note = _get_context_note_for_index(index_note)
    current_data = _parse_existing_context_note(context_note.text()) if context_note.exists() else {}
    all_names = [n.name for n in all_associated]

    for i, note in enumerate(all_associated):
        # Only process if it's in our target list and not already extracted
        if (note in target_set) and (note.name not in current_data):
            print(f"[{i+1}/{len(all_associated)}] Extracting: {note.name}...")
            
            # Extract context and capture usage metadata
            res, usage = _process_single_note_context(
                note, model, current_data, all_names, context_selector,
                max_context=max_context, 
                system_prompt=system_prompt, 
                config=config, 
                verbose=verbose,
                return_usage=True # Crucial for the sleep logic
            )
            
            if res:
                # Update data and write to disk immediately (checkpointing)
                current_data[note.name] = res
                context_note.write(_format_context_extraction(all_associated, current_data))
                
                # --- Throttling Logic ---
                wait = _get_sleep_time_from_usage(usage, sleep_threshold, default_sleep)
                if wait > 0:
                    if verbose or wait > default_sleep:
                        status = "Rate limit low" if wait > default_sleep else "Pacing"
                        print(f"  ({status}) Sleeping {wait:.2f}s...")
                    time.sleep(wait)
            else:
                print(f"  -> Skipping '{note.name}' (no response).")

    print(f"Successfully processed context for: {index_note.name}")

In [ ]:
# #| export

# def create_context_extraction_for_index_note(
#     index_note: VaultNote,
#     model: 'SupportedLLM',
#     info_notes: Optional[List[VaultNote]] = None,
#     max_context: int = 4096,
#     system_prompt: Optional[str] = CONTEXT_EXTRACTION_SYSTEM_PROMPT,
#     config: Optional[dict] = None,
#     verbose: bool = False,
#     context_selector: Optional[ContextSelector] = None,
# ) -> None:
#     """Generates and saves a context extraction file for a given index note."""
#     context_selector = context_selector or _default_context_selector
#     all_associated = _resolve_info_notes_for_index(index_note)
#     if not all_associated:
#         return print(f"Warning: No info notes found for '{index_note.name}'.")

#     target_set = set(info_notes) if info_notes else set(all_associated)
#     context_note = _get_context_note_for_index(index_note)
#     current_data = _parse_existing_context_note(context_note.text()) if context_note.exists() else {}
    
#     all_names = [n.name for n in all_associated]
#     for i, note in enumerate(all_associated):
#         if (note in target_set) and (note.name not in current_data):
#             print(f"[{i+1}/{len(all_associated)}] Extracting context for '{note.name}'...")
#             res = _process_single_note_context(
#                 note, model, current_data, all_names, context_selector,
#                 max_context=max_context, system_prompt=system_prompt, config=config, verbose=verbose
#             )
#             if res:
#                 current_data[note.name] = res
#                 context_note.write(_format_context_extraction(all_associated, current_data))
#             else:
#                 print(f"  -> No context returned for '{note.name}'.")

#     print(f"Context extraction completed: {context_note.name}")

In [ ]:
import unittest
from unittest.mock import MagicMock, patch, call
from pathlib import Path

class TestContextExtraction(unittest.TestCase):

    def setUp(self):
        # Common Mocks
        self.mock_vault = MagicMock()
        self.mock_model = MagicMock()
        
        # Mock Index Note
        self.index_note = MagicMock()
        self.index_note.name = "_index_Algebra"
        self.index_note.rel_path = "Algebra/_index_Algebra.md"
        self.index_note.vault = self.mock_vault
        # Mock pathlib behavior for parent directory
        self.index_note.path = Path("Algebra/_index_Algebra.md")
        
        # Mock Info Notes
        self.note_a = MagicMock()
        self.note_a.name = "Definition A"
        self.note_a.text.return_value = "Content of A"
        
        self.note_b = MagicMock()
        self.note_b.name = "Theorem B"
        self.note_b.text.return_value = "Content of B"

    def test_format_llm_input(self):
        """
        Verifies that the previous context and new passage are combined correctly.
        """
        # Call the function directly from the global scope (__main__)
        prev = "K is a field."
        new_text = "Let V be a vector space over K."
        
        # Assuming _format_llm_input is defined in the notebook cells above
        result = _format_llm_input(prev, new_text)
        
        self.assertIn("PREVIOUS CONTEXT:\nK is a field.", result)
        self.assertIn("NEW PASSAGE:\nLet V be a vector space over K.", result)

    def test_parse_existing_context_note(self):
        """
        Verifies that the markdown parser correctly extracts note-context pairs.
        """
        sample_text = (
            "- [[Definition A]]\n"
            "    - Context for A line 1\n"
            "      Context for A line 2\n"
            "- [[Theorem B]]\n"
            "    - Context for B"
        )
        
        # Call the function directly
        result = _parse_existing_context_note(sample_text)
        
        self.assertEqual(result['Definition A'], "Context for A line 1\nContext for A line 2")
        self.assertEqual(result['Theorem B'], "Context for B")

    # Patching objects in __main__ because we are running inside the notebook
    @patch('__main__.VaultNote')
    @patch('__main__._resolve_info_notes_for_index')
    @patch('__main__.process_standard_information_note')
    @patch('__main__.MarkdownFile')
    @patch('__main__.extract_context_with_lm')
    @patch('__main__._format_context_extraction')
    def test_context_accumulation_flow(self, mock_format, mock_extract, mock_md_file, mock_process, mock_resolve, mock_vault_note_cls):
        """
        CRITICAL TEST: Verifies that context generated for Note A is passed 
        as 'Previous Context' when processing Note B.
        """
        # 1. Setup Dependencies
        mock_resolve.return_value = [self.note_a, self.note_b]
        
        # Mock the text processing of the notes
        mock_process.side_effect = ["Processed Text A", "Processed Text B"]
        
        # Mock the LLM responses
        # First call (Note A) returns "Context A"
        # Second call (Note B) returns "Context B"
        mock_extract.side_effect = ["Context A", "Context B"]
        
        # Mock the Context Note file creation
        mock_context_note = MagicMock()
        mock_context_note.exists.return_value = False # File doesn't exist yet
        mock_vault_note_cls.return_value = mock_context_note

        # 2. Execute
        create_context_extraction_for_index_note(self.index_note, self.mock_model)

        # 3. Verify LLM Calls
        # We expect 2 calls to the LLM.
        self.assertEqual(mock_extract.call_count, 2)
        
        # Check arguments for Note A (First Call)
        args_a, _ = mock_extract.call_args_list[0]
        input_text_a = args_a[1] # 0 is model, 1 is text
        self.assertIn("NEW PASSAGE:\nProcessed Text A", input_text_a)
        
        # Check arguments for Note B (Second Call)
        # CRITICAL: The input text MUST contain "Context A" (the result of the previous step)
        args_b, _ = mock_extract.call_args_list[1]
        input_text_b = args_b[1]
        self.assertIn("PREVIOUS CONTEXT:\nContext A", input_text_b)
        self.assertIn("NEW PASSAGE:\nProcessed Text B", input_text_b)

        # 4. Verify Write Calls
        # Should write twice (incremental saving)
        self.assertEqual(mock_context_note.write.call_count, 2)

    @patch('__main__.VaultNote')
    @patch('__main__._resolve_info_notes_for_index')
    @patch('__main__.process_standard_information_note')
    @patch('__main__.MarkdownFile')
    @patch('__main__.extract_context_with_lm')
    @patch('__main__._parse_existing_context_note')
    @patch('__main__._format_context_extraction')
    def test_incremental_update_uses_existing_context(self, mock_format, mock_parse, mock_extract, mock_md_file, mock_process, mock_resolve, mock_vault_note_cls):
        """
        Verifies that if Note A already has context in the file, we skip LLM for A,
        but still pass A's context to Note B.
        """
        # 1. Setup Dependencies
        mock_resolve.return_value = [self.note_a, self.note_b]
        
        # Mock existing data in the file
        mock_context_note = MagicMock()
        mock_context_note.exists.return_value = True
        mock_context_note.text.return_value = "Raw Markdown Content"
        mock_vault_note_cls.return_value = mock_context_note
        
        # Parser returns existing context for Note A
        mock_parse.return_value = {self.note_a.name: "Existing Context A"}
        
        # Mock processing for Note B (Note A shouldn't be processed)
        mock_process.return_value = "Processed Text B"
        
        # Mock LLM for Note B
        mock_extract.return_value = "New Context B"

        # 2. Execute
        create_context_extraction_for_index_note(self.index_note, self.mock_model)

        # 3. Verify Logic
        # LLM should be called ONLY ONCE (for Note B)
        self.assertEqual(mock_extract.call_count, 1)
        
        # Verify the input to that single call
        args, _ = mock_extract.call_args
        input_text = args[1]
        
        # CRITICAL: Even though we didn't run LLM on Note A, its "Existing Context A"
        # must be present in the prompt for Note B.
        self.assertIn("PREVIOUS CONTEXT:\nExisting Context A", input_text)
        self.assertIn("NEW PASSAGE:\nProcessed Text B", input_text)
        
        # Verify we wrote the file (updating with B)
        self.assertTrue(mock_context_note.write.called)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

E.E.
ERROR: test_context_accumulation_flow (__main__.TestContextExtraction.test_context_accumulation_flow)
CRITICAL TEST: Verifies that context generated for Note A is passed
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\hyunj\AppData\Local\Programs\Python\Python312\Lib\unittest\mock.py", line 1396, in patched
    return func(*newargs, **newkeywargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hyunj\AppData\Local\Temp\ipykernel_10108\3719141125.py", line 90, in test_context_accumulation_flow
    create_context_extraction_for_index_note(self.index_note, self.mock_model)
  File "C:\Users\hyunj\AppData\Local\Temp\ipykernel_10108\3042671179.py", line 39, in create_context_extraction_for_index_note
    res, usage = _process_single_note_context(
    ^^^^^^^^^^
ValueError: too many values to unpack (expected 2)

ERROR: test_incremental_update_uses_existing_context (__main__.TestContextExtraction.test_in

[1/2] Extracting: Definition A...
[2/2] Extracting: Theorem B...


In [ ]:
#| export
def get_context_extraction_map(vault, reference: str) -> Dict[str, str]:
    """
    Locates and parses the context extraction note for a given reference.
    
    Args:
        vault: The Vault object.
        reference: The reference string (e.g., 'Algebra').
        
    Returns:
        A dictionary mapping note names to their extracted context strings.
        Returns an empty dict if the context note does not exist.
    """
    # Construct the expected name of the context extraction note
    context_note_name = f"_context_extraction_{reference}"
    
    # Find the note in the vault
    context_note = VaultNote(vault, name=context_note_name)
    
    if not context_note.exists():
        # print(f"Debug: Context extraction note '{context_note_name}' not found.")
        return {}
        
    # Use the existing helper to parse the markdown content
    return _parse_existing_context_note(context_note.text())


In [ ]:
#| export
def get_accumulated_context_for_notes(
    info_notes: List[VaultNote], 
    reference: str
) -> str:
    """
    Retrieves and accumulates the context extraction text for a list of notes.
    
    This function looks up the context for each note in the provided list
    and joins them into a single string, suitable for passing to an LLM.
    
    Args:
        info_notes: The list of VaultNote objects to retrieve context for.
        reference: The reference string used to locate the context file.
        
    Returns:
        A single string containing the combined context for the found notes,
        separated by double newlines.
    """
    if not info_notes:
        return ""
        
    vault = info_notes[0].vault
    
    # 1. Get the master dictionary of context data
    context_map = get_context_extraction_map(vault, reference)
    
    accumulated_parts = []
    
    # 2. Iterate through the requested notes in order
    for note in info_notes:
        if note.name in context_map:
            context_text = context_map[note.name]
            # Only add non-empty context
            if context_text.strip():
                accumulated_parts.append(context_text)
            
    return "\n\n".join(accumulated_parts)

In [ ]:
#| hide
@patch('__main__.VaultNote')
@patch('__main__._parse_existing_context_note')
def test_accumulation_retrieval(mock_parse, mock_vault_note_cls):
    mock_vault = MagicMock()
    mock_context_note = MagicMock()
    mock_context_note.exists.return_value = True
    mock_context_note.text.return_value = "dummy"
    mock_vault_note_cls.return_value = mock_context_note
    
    # Setup parsed map
    mock_parse.return_value = {"Note1": "Context 1", "Note2": "Context 2"}
    
    note_obj = MagicMock(name="Note1")
    note_obj.name = "Note1"
    note_obj.vault = mock_vault
    
    # Test Retrieval
    accumulated = get_accumulated_context_for_notes([note_obj], "Algebra")
    
    test_eq(accumulated, "Context 1")
    # Verify naming convention
    mock_vault_note_cls.assert_called_with(mock_vault, name="_context_extraction_Algebra")

test_accumulation_retrieval()

In [ ]:
#| hide

def test_parse_reset_time():
    """Verifies parsing of different API reset formats."""
    # Duration strings (OpenAI/Groq)
    test_eq(_parse_reset_time("1.5s"), 1.5)
    test_eq(_parse_reset_time("2m0s"), 120.0)
    test_eq(_parse_reset_time("1h1m1s"), 3600 + 60 + 1)
    
    # Unix Timestamps (Anthropic)
    future_now = time.time() + 10
    test_eq(round(_parse_reset_time(str(future_now))), 10)
    
    # Simple integers
    test_eq(_parse_reset_time("10"), 10.0)
    test_eq(_parse_reset_time(None), 0.0)

def test_get_sleep_time_from_usage_fixed():
    # Case 3: Anthropic style headers
    anthropic = {
        'anthropic-ratelimit-tokens-remaining': 10,
        'anthropic-ratelimit-tokens-limit': 1000,
        'anthropic-ratelimit-tokens-reset': '2.0s'
    }
    result = _get_sleep_time_from_usage(anthropic, default_sleep=0)
    print(f"Result: {result}")
    test_eq(result, 2.5)


#| hide
@patch('time.sleep')
@patch('__main__._process_single_note_context')
@patch('__main__._resolve_info_notes_for_index')
@patch('__main__._get_context_note_for_index')
def test_pacing_in_main_loop(mock_get_note, mock_resolve, mock_process, mock_sleep):
    """Verifies that the main loop actually calls sleep between notes."""
    # Setup 2 notes
    n1, n2 = MagicMock(name="N1"), MagicMock(name="N2")
    n1.name, n2.name = "Note1", "Note2"
    mock_resolve.return_value = [n1, n2]
    
    # Mock context note
    mock_context = MagicMock()
    mock_context.exists.return_value = False
    mock_get_note.return_value = mock_context

    # Mock LLM response and usage
    # Note 1 returns healthy usage, Note 2 is already in data (skipped)
    usage = {'x-ratelimit-remaining-tokens': 1000, 'x-ratelimit-limit-tokens': 1000}
    mock_process.return_value = ("Context Result", usage)

    # Execute
    create_context_extraction_for_index_note(
        MagicMock(), MagicMock(), default_sleep=2.5
    )

    # Verify sleep was called with the default_sleep value
    mock_sleep.assert_any_call(2.5)

test_parse_reset_time()
test_get_sleep_time_from_usage_fixed()
test_pacing_in_main_loop()

Result: 2.5
[1/2] Extracting: Note1...
[2/2] Extracting: Note2...
Successfully processed context for: <MagicMock name='mock.name' id='2494912642848'>
